# Mô hình và đồ thị nhân quả — toàn bộ quy trình, từ dữ liệu thô đến phán quyết

Notebook này gom **mọi thứ liên quan tới mô hình và đồ thị nhân quả** của dự án vào một chỗ: dữ liệu, cách
**xây dựng** đồ thị, cách **huấn luyện** từng cơ chế, cách suy luận can thiệp, và cách đánh giá. Mỗi bước có lời
giải thích rồi code **chạy lại chính module trong `src/`** trên file thật — không chép số tay.

**Cách chạy:** mở trong IDE, chọn kernel **`.venv`** (`.venv\Scripts\python.exe`), rồi *Run All* (≈ 1 phút).
Notebook chỉ **đọc** dữ liệu, không ghi đè file nào, không chạy tải. Đừng chạy khi đang đo ramp.

## Dự án có HAI nhánh mô hình — đừng lẫn

| | **Phần A — Global Causal DAG** | **Phần B — Bộ dự đoán khả thi** |
|---|---|---|
| Code | `src/agents/capacity_agent.py` + `src/scm/*` | `src/scm/feasibility_predictor.py` + `src/agents/feasibility_agent.py` |
| Câu hỏi | Tăng workload ở gateway thêm δ% thì CPU/mem/độ trễ **mọi node** đổi bao nhiêu? | Thêm tính năng **chưa tồn tại** thì còn đạt SLO ở tải L không, node nào nghẽn, **điểm gãy** ở đâu? |
| Dữ liệu huấn luyện | RCAEval **RE2-SS** (đoạn bình thường trước khi tiêm lỗi) | `SS-TRAIN` — testbed Sock Shop tự dựng, quét tải |
| Đồ thị | 28 node (workload/cpu/mem/latency × 7 service), cạnh cấu trúc + cạnh **học** | Chuỗi đơn giản L → W_s → CPU_s → u_s, cạnh theo chuỗi gọi |
| Cơ chế | DoWhy GCM: tuyến tính hệ số không âm + hồi quy hàng đợi | NNLS `CPU = α + β·W` + ngưỡng u* |
| Đánh giá trong bài | RQ1–RQ4 | RQ5 (tiến cứu, n = 10) |

| Mục | Nội dung |
|---|---|
| **A1** | Dữ liệu RE2-SS và *cổng tín hiệu* |
| **A2** | Xây đồ thị: đồ thị phụ thuộc → bảng mẫu cạnh → cạnh cấu trúc |
| **A3** | Học cạnh: ứng viên → chấm held-out → knee-point → kiểm OOD |
| **A4** | Gán cơ chế và huấn luyện DAG |
| **A5** | Đường nhanh: 21 mô hình hai biến và giao thức OOD 67/33 |
| **A6** | Suy luận can thiệp: forward tất định so với Monte Carlo |
| **A7** | Bao an toàn chứng nhận (Mệnh đề 2) và can thiệp đồng thời |
| **A8** | Độ ổn định, ảnh hưởng người dùng, biên dự phòng |
| **B1–B8** | Bộ dự đoán khả thi: bài toán → dữ liệu → đồ thị → **huấn luyện** → đóng băng → dự đoán → tiến cứu → chẩn đoán |
| **C** | Hạn chế đã biết của cả hai nhánh và hướng cải tiến |

In [ ]:
import glob, json, os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

ROOT = Path.cwd()
while not (ROOT / 'src').exists():          # notebook nam trong notebooks/ -> leo len goc repo
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / 'src' / 'scm'), str(ROOT / 'src' / 'agents'), str(ROOT / 'src'),
                str(ROOT / 'experiments' / 'collect'), str(ROOT / 'experiments' / 'feasibility')]
import contextlib, io, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TQDM_DISABLE', '1')

@contextlib.contextmanager
def quiet():
    '''an log dai cua dowhy/agent de notebook de doc'''
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        yield

import feasibility_predictor as FP                  # bo du doan (nguon su that duy nhat)
from request_router import SOCKSHOP_CALL_CHAINS     # taxonomy archetype
import load_sweep_collect as HARNESS                # bang FEATURES cua harness (anchor = Delta)

FROZEN = ROOT / 'data' / 'processed' / 'frozen'
RAW = ROOT / 'data' / 'raw'
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
try:
    display
except NameError:            # chay ngoai Jupyter (kiem thu) -> in thuong
    display = print
print('goc repo:', ROOT)

---
# PHẦN A — Global Causal DAG (`CapacityAgent`)

Đây là mô hình nhân quả "đầy đủ": mỗi service có 4 biến (workload, CPU, bộ nhớ, độ trễ p50) và đồ thị mô tả biến
nào gây ra biến nào. Can thiệp `do(front-end_workload = w)` rồi lan truyền theo thứ tự tô-pô để biết từng node đổi
bao nhiêu.

## A1. Dữ liệu: RCAEval RE2-SS

RE2-SS là bộ dữ liệu benchmark phân tích nguyên nhân gốc (RCA): mỗi *kịch bản* tiêm một lỗi (CPU, mem, delay, loss,
disk, socket) vào một service, lặp 3 lần. Mỗi lần có `simple_metrics.csv` (mỗi giây một dòng) và `inject_time.txt`.
`load_multi_service_data()` **chỉ lấy đoạn TRƯỚC thời điểm tiêm lỗi** → dữ liệu vận hành bình thường.

In [ ]:
import data_processor as DP
df_re2 = DP.load_multi_service_data(str(RAW), services=DP.SERVICES)
scen = sorted(p.name for p in RAW.iterdir() if p.is_dir() and not p.name.startswith('SS-') and p.name != 'trainticket')
print(f'{len(scen)} kich ban loi: {scen}')
print(f'du lieu binh thuong gop lai: {df_re2.shape[0]} giay x {df_re2.shape[1]} cot')
print('cac metric moi service:', sorted({c.split("_", 1)[1] for c in df_re2.columns}))
display(df_re2.describe().T[['mean', 'std', 'min', 'max']].round(3).head(14))

### A1.1 Cổng tín hiệu — dữ liệu có đủ để học quan hệ workload → CPU không?

Một mô hình chỉ học được quan hệ nhân quả nếu **nguyên nhân thật sự biến thiên**. Ô dưới so R² của hồi quy
`CPU ~ workload` trên RE2-SS với trên testbed tự dựng `SS-TRAIN` (quét tải 10 → 250 req/s). Đây là phát hiện
"benchmark không đủ tín hiệu" của bài (RQ3): trên RE2-SS workload gần như **không đổi**, nên sai số MAPE thấp
không chứng minh gì — một hằng số cũng đạt được.

In [ ]:
import feasibility_predictor as FP
from sklearn.linear_model import LinearRegression
ss_train = FP.load_runs(str(RAW / 'SS-TRAIN'))
rows = []
for s in ['front-end', 'catalogue', 'user', 'carts', 'orders']:
    for name, d in (('RE2-SS', df_re2), ('SS-TRAIN', ss_train)):
        x = d[[f'{s}_workload']].dropna(); y = d.loc[x.index, f'{s}_cpu']
        ok = y.notna(); x, y = x[ok], y[ok]
        r2 = LinearRegression().fit(x, y).score(x, y)
        rows.append({'service': s, 'bo': name, 'workload TB': x.iloc[:, 0].mean(), 'workload CV': x.iloc[:, 0].std() / x.iloc[:, 0].mean(),
                     'R2 CPU~workload': r2})
sig = pd.DataFrame(rows).pivot(index='service', columns='bo', values=['workload CV', 'R2 CPU~workload'])
display(sig.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (name, d) in zip(axes, (('RE2-SS', df_re2), ('SS-TRAIN', ss_train))):
    ax.scatter(d['front-end_workload'], d['front-end_cpu'], s=2, alpha=.2)
    ax.set_title(f'{name}: front-end CPU theo workload'); ax.set_xlabel('workload (req/s)'); ax.set_ylabel('CPU')
plt.tight_layout(); plt.show()

**Hệ quả:** mọi con số "độ chính xác" của nhánh A trên RE2-SS phải đọc cùng cổng tín hiệu này. Đó là lý do nhánh B
phải tự dựng testbed có tải biến thiên thật.

## A2. Xây đồ thị

### A2.1 Bảng mẫu cạnh (`scm_graph_builder.DEFAULT_EDGE_TEMPLATES`)

Ý tưởng lấy từ CausIL: chất lượng đồ thị nhân quả được quyết định bởi **ràng buộc không gian tìm kiếm** bằng tri thức
miền, không phải thuật toán tìm kiếm. Chỉ cạnh thuộc một mẫu được khai báo mới **được phép** tồn tại; mọi cạnh khác
bị cấm.

- `intra` = trong cùng một service; `inter` = giữa hai service có quan hệ gọi.
- `forward` = cha là bên gọi; `reverse` = cha là bên bị gọi (độ trễ callee cộng vào caller).
- `learned=False` → cạnh **cấu trúc**, thêm thẳng. `learned=True` → phải **qua kiểm định** (A3).

In [ ]:
import scm_graph_builder as GB, scm_edge_selector as ES
GRAPH = str(ROOT / 'src/graph/sockshop_agent_graph.json')
display(pd.DataFrame([{'ten': t.name, 'cha': t.parent_metric, 'con': t.child_metric, 'pham vi': t.scope,
                       'chieu': t.direction, 'phai hoc?': t.learned} for t in GB.DEFAULT_EDGE_TEMPLATES]))

### A2.2 Cạnh cấu trúc (Tier 1 + Tier 2)

- **Tier 1** — `workload(caller) → workload(callee)` theo mọi cạnh gọi trong đồ thị phụ thuộc: tải lan theo chiều gọi.
- **Tier 2** — trong mỗi service, `workload → cpu`, `workload → mem`, `workload → latency-50`.

Chỉ tạo cạnh khi **cả hai cột có trong dữ liệu** (vd. service DB không có cột workload nên bị bỏ).

In [ ]:
struct = GB.structural_edges(GRAPH, DP.SERVICES, df_re2.columns, GB.DEFAULT_EDGE_TEMPLATES)
kind = lambda e: 'Tier1 workload->workload' if e[0].endswith('_workload') and e[1].endswith('_workload') else 'Tier2 ' + e[1].split('_', 1)[1]
print(f'{len(struct)} canh cau truc:')
display(pd.Series([kind(e) for e in struct]).value_counts().to_frame('so canh'))
print([e for e in struct if kind(e).startswith('Tier1')])

## A3. Học cạnh — quy trình 4 pha cho mỗi mẫu `learned=True`

Hai loại cạnh phải học: **CPU backpressure** (`cpu(caller) → cpu(callee)`) và **lan truyền độ trễ kiểu CausIL**
(`latency(callee) → latency(caller)`).

1. **Ứng viên:** mọi cặp gọi trong đồ thị phụ thuộc có đủ cột.
2. **Chấm điểm HELD-OUT:** sắp dữ liệu theo workload của service con, **train 67% tải thấp, test 33% tải cao**. So
   mô hình con `~ workload` (cơ sở) với `~ workload + cha ứng viên` (mở rộng). Điểm = mức tăng R² **trên tập tải
   cao chưa thấy**. *Vì sao không in-sample:* R²/BIC in-sample từng chọn cạnh làm MAPE held-out tăng 298% → 499%.
   Cạnh có hệ số mới bằng 0 bị loại (hệ số 0 vẫn có thể cho MAPE đẹp).
3. **Knee-point:** sắp mức tăng giảm dần, cắt ở điểm khuỷu tay thay vì một ngưỡng cố định.
4. **An toàn OOD:** can thiệp `do(gateway_workload = base·(1+δ))` với δ ∈ {5…300}% **có và không có** cạnh; loại
   cạnh nào làm **đảo dấu** (tải tăng mà CPU dự đoán giảm) ở service chưa từng đảo dấu.

In [ ]:
edge_reports = {}
for tpl in [t for t in GB.DEFAULT_EDGE_TEMPLATES if t.learned]:
    rep = GB.select_edges_for_template(tpl, GRAPH, df_re2, DP.SERVICES)      # pha 1-3 (chua OOD)
    edge_reports[tpl.name] = rep
    print(f"\n=== {tpl.name}: {len(rep['candidate_edges'])} ung vien -> {len(rep['selected_edges'])} qua held-out + knee-point")
    display(rep['scores'][['parent_col', 'child_col', 'r2_base_heldout', 'r2_ext_heldout', 'r2_gain_heldout',
                           'mape_base_heldout', 'mape_ext_heldout']])

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, (name, rep) in zip(axes, edge_reports.items()):
    sc = rep['scores']; sel = set(rep['selected_edges'])
    ax.bar(range(len(sc)), sc['r2_gain_heldout'],
           color=['#2ca02c' if (p, c) in sel else '#bbb' for p, c in zip(sc.parent_col, sc.child_col)])
    ax.set_xticks(range(len(sc)), [f'{p.split("_")[0]}→{c.split("_")[0]}' for p, c in zip(sc.parent_col, sc.child_col)],
                  rotation=45, fontsize=7)
    ax.set_title(f'{name}: xanh = duoc chon (truoc diem khuyu)', fontsize=9); ax.set_ylabel('R2 tang (held-out)')
plt.tight_layout(); plt.show()

### A3.1 Pha 4 — kiểm an toàn OOD (cần fit DAG hai lần mỗi mẫu cạnh)

`CapacityAgent._build_extended_dag_fn()` dựng một DAG = cạnh cấu trúc + cạnh ứng viên, gán **đúng** quy tắc cơ chế
như DAG sản phẩm, rồi fit. Ô dưới chạy pha 4 cho cạnh CPU backpressure (~10 giây).

In [ ]:
from capacity_agent import CapacityAgent
with quiet():
    ag = CapacityAgent()                       # huan luyen day du (A4) -- can de dung build_model_fn
build_fn = ag._build_extended_dag_fn(nx.DiGraph(struct), df_re2)
sel_cpu = edge_reports['cpu_backpressure']['selected_edges']
with quiet():
    ood = ES.validate_ood_safety(build_fn, df_re2, sel_cpu, DP.SERVICES, metric='cpu', injection_services=[ag.default_injection])
pv = ood.pivot_table(index=['service'], columns=['variant', 'delta_pct'], values='cpu_change_pct')
display(pv.round(1))
inv = ood[ood.sign_inverted].groupby('variant').size()
print('so o DAO DAU (tai tang ma CPU du doan giam):', inv.to_dict())
print('canh CPU cuoi cung trong DAG san pham:', [e for e in ag._last_edge_build['learned_edges'] if e[0].endswith('_cpu')])

Số ô đảo dấu **bằng nhau** có và không có cạnh → cạnh mới không gây thêm đảo dấu nào → cả 3 cạnh được giữ. Các ô
đảo dấu sẵn có đến từ cơ chế nền (service có hiệu ứng gần 0 và nhiễu), không từ cạnh ứng viên.

Kết quả chọn cạnh được **cache** trong `data/processed/scm_cache/` (khoá = hash của đồ thị + dữ liệu + bảng mẫu), vì
pha 4 là phần đắt nhất; lần fit cuối cùng luôn chạy lại nên mô hình luôn tươi.

## A4. Gán cơ chế và huấn luyện DAG

`train_accurate_path()` ghép toàn bộ cạnh, **phá chu trình** nếu có (bỏ cạnh cuối của chu trình), lấy mẫu tối đa
2000 dòng, rồi gán cơ chế cho từng node (`_assign_dag_mechanisms`):

| Node | Cơ chế | Lý do |
|---|---|---|
| gốc (`front-end_workload`) | phân phối thực nghiệm | không có cha |
| `*_workload` có cha, `*_cpu`, `*_mem` | `LinearRegression(positive=True)` + nhiễu cộng | hệ số **không âm** → đơn điệu → Mệnh đề 2 dùng được |
| `*_latency-50` | `QueueingLatencyRegressor` + nhiễu cộng | M/M/1: độ trễ bùng nổ khi W → C |

**Hồi quy hàng đợi:** ước lượng trần `C = 1,5 × P99(W)` từ dữ liệu, biến đổi `W/(C − W)`, rồi hồi quy tuyến tính
**không âm** trên `[W, W/(C−W)]`. Hệ số âm trước đây làm 7/7 cơ chế độ trễ vi phạm tính đơn điệu; ép không âm vừa
sửa đơn điệu vừa **giảm** MAPE held-out.

In [ ]:
g_dag = ag.dag_graph
print(f'DAG san pham: {g_dag.number_of_nodes()} node, {g_dag.number_of_edges()} canh '
      f'(cau truc {len(ag._last_edge_build["structural_edges"])} + hoc {len(ag._last_edge_build["learned_edges"])})')
from queueing_regressor import QueueingLatencyRegressor
rows = []
for n in nx.topological_sort(g_dag):
    parents = sorted(g_dag.predecessors(n))
    if not parents:
        rows.append({'node': n, 'cha': '(goc)', 'co che': 'thuc nghiem', 'he so': '', 'chan (intercept)': ''}); continue
    sk = ag.global_dag_model.causal_mechanism(n).prediction_model.sklearn_model
    m = sk.model_ if isinstance(sk, QueueingLatencyRegressor) else sk
    rows.append({'node': n, 'cha': ', '.join(parents), 'co che': type(sk).__name__,
                 'he so': np.round(m.coef_, 4).tolist(), 'chan (intercept)': round(float(m.intercept_), 4)})
display(pd.DataFrame(rows))

In [ ]:
layer = {'workload': 0, 'cpu': 1, 'mem': 2, 'latency-50': 3}
svc_x = {s: i for i, s in enumerate(DP.SERVICES)}
pos = {n: (svc_x[n.split('_')[0]], -layer[n.split('_', 1)[1]]) for n in g_dag}
learned = set(map(tuple, ag._last_edge_build['learned_edges']))
fig, ax = plt.subplots(figsize=(13, 5.5))
nx.draw_networkx_nodes(g_dag, pos, ax=ax, node_size=900, node_color=[['#1f77b4', '#d62728', '#9467bd', '#2ca02c'][layer[n.split('_', 1)[1]]] for n in g_dag])
nx.draw_networkx_labels(g_dag, pos, {n: n.replace('_', '\n') for n in g_dag}, font_size=6, font_color='white', ax=ax)
nx.draw_networkx_edges(g_dag, pos, edgelist=[e for e in g_dag.edges if e not in learned], ax=ax, edge_color='#999', arrowsize=8,
                       connectionstyle='arc3,rad=0.12')
nx.draw_networkx_edges(g_dag, pos, edgelist=[e for e in g_dag.edges if e in learned], ax=ax, edge_color='orange', width=2.2,
                       arrowsize=12, connectionstyle='arc3,rad=0.3')
ax.set_title('Global Causal DAG: hang = loai metric (workload / cpu / mem / latency), cot = service; cam = canh HOC'); ax.axis('off')
plt.tight_layout(); plt.show()

Quan sát quan trọng từ hình: **không có cạnh nào từ CPU/mem sang độ trễ**. Cạnh backpressure là CPU → CPU và dừng
ở CPU, nên mô hình chưa diễn tả được "CPU bão hoà làm tăng độ trễ" — đúng cơ chế mà nhánh B thấy làm vỡ SLO.

## A5. Đường nhanh — 21 mô hình hai biến và giao thức OOD 67/33

`train_fast_path()` fit riêng `Workload → Target` cho mỗi (service × {CPU, Mem, Socket}). Giao thức đánh giá dùng
chung toàn dự án: **sắp theo workload, train 67% tải thấp, test 33% tải cao** — vì câu hỏi thật luôn là ngoại suy
lên tải cao hơn.

In [ ]:
display(ag.accuracy_df[['service', 'metric', 'mape_pct', 'r2', 'tag', 'n_train', 'n_test']])

Chú ý R² **âm** ở nhiều dòng dù MAPE thấp: trên tập test tải cao, mô hình còn kém hơn dự đoán bằng trung bình —
MAPE thấp chỉ vì CPU gần như không đổi (xem cổng tín hiệu A1.1). Đây là ví dụ cụ thể của "đánh giá rỗng".

## A6. Suy luận can thiệp

Can thiệp luôn ở **gateway** — node gốc, không có cha. Với node gốc, `do(X = x)` trùng với điều kiện hoá `X = x`
(không có cạnh nào để cắt), nên toán tử `do()` không cho lợi thế gì so với hồi quy thường; giá trị nằm ở **đồ thị**
(tải đổ vào node nào). Vì vậy dự án dùng **forward tất định**: đi theo thứ tự tô-pô, gọi `.predict()` của từng cơ chế
trên giá trị điểm của cha, không lấy mẫu nhiễu.

In [ ]:
from dowhy import gcm
from deterministic_forward import deterministic_forward
inj = f'{ag.default_injection}_workload'
base = float(ag.df_baseline[inj].mean()); target = base * 1.2
det, breached = deterministic_forward(g_dag, ag.global_dag_model, ag.df_baseline, {inj: target})
det0, _ = deterministic_forward(g_dag, ag.global_dag_model, ag.df_baseline, {inj: base})     # chinh mo hinh o delta = 0
np.random.seed(0)
with quiet():
    mc = gcm.interventional_samples(ag.global_dag_model, interventions={inj: lambda x: target}, num_samples_to_draw=2000)
cmp_ = pd.DataFrame({'du lieu (TB)': ag.df_baseline.mean(), 'mo hinh o delta=0': pd.Series(det0),
                     'forward tat dinh': pd.Series(det), 'Monte Carlo (TB 2000 mau)': mc.mean()})
cmp_['% doi so voi du lieu'] = 100 * (cmp_['forward tat dinh'] / cmp_['du lieu (TB)'] - 1)
cmp_['% doi so voi mo hinh'] = 100 * (cmp_['forward tat dinh'] / cmp_['mo hinh o delta=0'] - 1)
cmp_['lech tat dinh vs MC %'] = 100 * (cmp_['forward tat dinh'] / cmp_['Monte Carlo (TB 2000 mau)'] - 1)
print(f'do({inj}: {base:.2f} -> {target:.2f}, +20%)   node vuot tran hang doi: {breached}')
display(cmp_.drop(index=inj).round(3).sort_values('% doi so voi mo hinh', ascending=False))
print('so node co % doi AM khi tai TANG -- so voi du lieu:', int((cmp_['% doi so voi du lieu'] < -0.5).sum()),
      '| so voi mo hinh:', int((cmp_['% doi so voi mo hinh'] < -0.5).sum()))

In [ ]:
with quiet():
    sim = ag.simulate_intervention(ag.default_injection, 20)
display(pd.DataFrame(sim).T[['n_hops', 'cpu_change_pct', 'cpu_confidence', 'mem_change_pct', 'latency_change_pct', 'latency_confidence']])

**Đọc hai bảng trên:**

- Forward tất định khớp Monte Carlo trong vài phần trăm ở node tuyến tính; lệch lớn hơn ở node độ trễ (hàm lồi —
  trung bình của hàm ≠ hàm của trung bình). Dự án chọn forward tất định vì không có nhiễu lấy mẫu và nhanh ~14×.
- ⚠️ `simulate_intervention()` tính `% đổi` **so với trung bình dữ liệu**, không so với chính mô hình ở δ = 0. Mô hình
  fit trên mẫu 2000 dòng nên giá trị dự đoán ở δ = 0 đã lệch trung bình dữ liệu; kết quả là vài node (vd.
  `payment_cpu`, `shipping_latency-50`) **báo giảm khi tải tăng**. So với mô hình ở δ = 0 thì không node nào giảm —
  đúng tính đơn điệu. Đây là lỗi cách báo cáo, không phải lỗi cơ chế (Phần C, A-8).
- `*_confidence` lấy từ độ ổn định của **chính cơ chế đó** (A8): `unstable` nghĩa là không nên tin con số ở cột đó.

## A7. Bao an toàn chứng nhận (Mệnh đề 2) và can thiệp đồng thời (Mệnh đề 3)

**Mệnh đề 2:** nếu **mọi** cơ chế đơn điệu không giảm theo từng cha (hệ số ≥ 0), thì với **mọi** δ ∈ [δ_min, δ_max],
giá trị điểm của mỗi node nằm giữa hai lần forward ở δ_min và δ_max. Hai lần forward là đủ — không cần quét.
Tiền điều kiện được **kiểm lại lúc chạy** (`_check_monotone_precondition`), không tin mù vào ràng buộc của solver;
vi phạm thì từ chối cấp chứng nhận.

In [ ]:
print('vi pham tien dieu kien don dieu:', ag._check_monotone_precondition() or 'khong co')
env = ag.certified_envelope(ag.default_injection, 5, 50)
envdf = pd.DataFrame({k: v for k, v in env.items() if isinstance(v, dict)}).T
display(envdf.round(3))

# kiem chung bang cach quet: moi delta trong [5, 50] phai nam trong [lo, hi]
deltas = np.linspace(5, 50, 19); viol = 0
for dlt in deltas:
    vals, _ = deterministic_forward(g_dag, ag.global_dag_model, ag.df_baseline, {inj: base * (1 + dlt / 100)})
    viol += sum(not (envdf.loc[n, 'lo'] - 1e-9 <= vals[n] <= envdf.loc[n, 'hi'] + 1e-9) for n in envdf.index)
print(f'quet {len(deltas)} gia tri delta x {len(envdf)} node: {viol} lan nam NGOAI bao')

**Mệnh đề 3 — nhiều yêu cầu cùng lúc:** bao đồng thời vẫn chỉ cần hai lần forward. Nhưng **không cộng dồn được**:
cơ chế độ trễ lồi (`W/(C−W)`), nên tác động thật của hai yêu cầu dồn vào cùng một nút thắt **lớn hơn** tổng hai tác
động riêng (bất đẳng thức Jensen). Ô dưới đo khoảng chênh "đồng thời − tổng ngây thơ" khi tăng tải cùng lúc ở gateway
và ở `orders`. (Bài đã **rút lại** khẳng định này trên dữ liệu thật Alibaba vì không đứng vững khi nhân đôi cửa sổ
quan sát; ở đây chỉ minh hoạ cơ chế.)

In [ ]:
jn = ag.compare_joint_vs_naive_sum([ag.default_injection, 'orders'], delta_pct=50.0)
jdf = pd.DataFrame(jn).T
jdf['gap %'] = 100 * jdf['gap'] / jdf['naive_sum'].abs()
jdf['nam duoi orders?'] = [nx.has_path(g_dag, 'orders_workload', n) for n in jdf.index]
display(jdf.round(4).reindex(jdf['gap %'].abs().sort_values(ascending=False).index).head(10))
print('node co gap != 0:', sorted(jdf.index[jdf['gap'].abs() > 1e-9]))
qcoef = {n: ag.global_dag_model.causal_mechanism(n).prediction_model.sklearn_model.model_.coef_.round(4).tolist()
         for n in g_dag if n.endswith('latency-50') and g_dag.in_degree(n) > 0}
print('he so cua co che do tre [W..., W/(C-W)...]:', qcoef)

**Đọc kết quả — có hai hiệu ứng khác nhau, đừng lẫn:**

1. **Tính lồi (Mệnh đề 3) không xuất hiện:** trên RE2-SS, số hạng hàng đợi `W/(C−W)` của mọi cơ chế độ trễ được fit ra
   hệ số **0**, nên không có siêu cộng dồn — đúng lý do khẳng định này bị rút khỏi phần dữ liệu thật.
2. **Khoảng chênh khác 0 chỉ nằm ở hạ nguồn `orders`** và **âm** (dưới cộng dồn). Nguyên nhân là cấu trúc: Sock Shop chỉ
   có **một** gateway, nên can thiệp thứ hai (`orders`) luôn là con cháu của `front-end`. `do(orders_workload)` **cắt**
   cạnh `front-end → orders`; tổng ngây thơ vì thế đếm trùng phần tải đi qua đường đó. Mệnh đề 3 giả định các can
   thiệp nằm ở **các node gốc khác nhau** — trên hệ một gateway, kịch bản đó không tồn tại.

## A8. Độ ổn định, ảnh hưởng người dùng và biên dự phòng

- `evaluate_node_stability()`: đánh giá **chính cơ chế đang dùng** (không fit lại) trên 33% giá trị cao nhất của node.
  Tag POOR → node đó bị đánh dấu `unstable` trong mọi kết quả.
- `rank_user_impact()`: độ co giãn — % thay đổi của một node độ trễ khi một node khác tăng 1%, qua forward tất định.
- `compute_headroom()`: đổi dự đoán thành **% biên còn lại** tới mức cao nhất đã quan sát (P99 của node) — vì trần năng
  lực vật lý không định danh được từ dữ liệu chưa bão hoà.

In [ ]:
with quiet():
    stab = ag.evaluate_node_stability(); imp = ag.rank_user_impact(); keys = ag.select_key_nodes()
display(stab)
print('Tag theo loai metric:'); display(pd.crosstab(stab.node.str.split('_', n=1).str[1], stab.tag))
print('\nTop anh huong nguoi dung (do co gian):'); display(imp.head(10))
print('\nselect_key_nodes:', {k: v for k, v in keys.items()})

In [ ]:
hr = ag.compute_headroom(sim)
display(pd.DataFrame({(s, m): v for s, d in hr.items() for m, v in d.items() if isinstance(v, dict)}).T
        [['baseline', 'projected', 'ceiling_p99', 'consumed_pct', 'remaining_pct', 'status']].round(3))

**Tóm tắt nhánh A:** đồ thị đứng vững dưới kiểm định can thiệp (cạnh cấu trúc + 5 cạnh học qua held-out và OOD),
chứng nhận đơn điệu được kiểm lúc chạy. Nhưng: (1) RE2-SS thiếu tín hiệu tải nên độ chính xác trên nó không chứng
minh gì; (2) nhiều cơ chế CPU/độ trễ bị gắn POOR; (3) chưa có cạnh CPU → độ trễ; (4) `do()` không cho lợi thế vì can
thiệp luôn ở gốc. Nhánh B được xây để trả lời câu hỏi khả thi trên dữ liệu **có** tín hiệu.

---
# PHẦN B — Bộ dự đoán khả thi cho tính năng chưa tồn tại

Phần này đi theo đúng thứ tự dữ liệu chảy qua bộ dự đoán, **kể cả quy trình huấn luyện và đóng băng**: bài toán →
dữ liệu → đồ thị → fit cơ chế → hiệu chỉnh u* → fit tham số P2 → probe `k` → đóng băng → dự đoán → tiến cứu → chẩn đoán.

## B1. Bài toán

Đội phát triển muốn thêm một **tính năng chưa hề tồn tại** (vd. *đăng nhập nhanh*) vào Sock Shop. Câu hỏi
**trước khi viết dòng code nào**: *với kiến trúc hiện có, hệ thống còn đạt SLO ở mức tải đỉnh L không, và
service nào nghẽn trước?*

- Vì tính năng chưa tồn tại nên **không có telemetry của chính nó**. Mô hình chỉ được học từ dữ liệu *không có
  tính năng* (baseline), rồi *can thiệp* thêm tải của tính năng vào. Đây là câu hỏi *what-if*, không phải dự báo
  chuỗi thời gian.
- **Phạm vi:** tính năng chỉ thêm traffic đi qua service và cạnh gọi **đã có**; không thêm service/cạnh mới.
- **Đáp án:** sau khi đóng băng dự đoán, cài tính năng thật rồi tăng tải từng bậc (*ramp*) tới khi vỡ SLO.
  Mức tải cuối còn đạt SLO gọi là **điểm gãy** `lo`, bậc đầu vi phạm là `hi`.

SLO và trần CPU được chốt trong file cấu hình, không cắm cứng trong code:

In [ ]:
slo = FP.load_slo()
limits = json.load(open(ROOT / 'deploy/sockshop/limits.json', encoding='utf-8'))
cores = {k: v for k, v in limits['configs']['RE2'].items()}
print(f"SLO: p99 <= {slo['p99_s']*1000:.0f} ms, ti le loi <= {slo['err_rate']*100:.0f}%")
display(pd.Series(cores, name='tran CPU (core), cau hinh RE2').to_frame())

Trần **RE2** chép lại giới hạn CPU của bộ dữ liệu RCAEval RE2-SS (0,5 core cho hầu hết service; `user` 0,2;
`payment` 0,1). Hai cấu hình thử khác (C1, C2) **thất bại** vì hạn ngạch nhỏ gây CFS throttling — SLO vỡ ở mức
sử dụng CPU rất thấp, ngoài phạm vi mô hình. Vì vậy mọi kết quả dưới đây chỉ ở RE2.

## B2. Dữ liệu

### B2.1 Các bộ dữ liệu và vai trò

| Thư mục `data/raw/` | Vai trò | Có tính năng? | Trần CPU |
|---|---|---|---|
| `SS-TRAIN` | **Huấn luyện** cơ chế ρ, α, β (quét tải 10→250 req/s) | Không | Không trần |
| `SS-LIMITS` (ramp `base`) | Hiệu chỉnh ngưỡng **u*** | Không | RE2 |
| `SS-LIMITS` (promo, recs) | Tập **phát triển** — fit tham số P2 (`c_s`, `x`) | Có | RE2 |
| `SS-LIMITS` (track, review) | Tập **khoá** — mở đúng một lần | Có | RE2 |
| `SS-LIMITS-CLEAN` | Tính năng do agent độc lập cài (hồi cứu) | Có | RE2 |
| **`SS-PROSP2`** | **Tiến cứu** n = 10, dự đoán đóng băng *trước* khi đo | Có | RE2 |
| `SS-VERIFY*`, `SS-PROSP2-R2/R3`, `SS-PROSP4*` | Kiểm hệ / nguồn đã gộp / chẩn đoán | — | RE2 |

Dữ liệu thô **không nằm trong git** (`.gitignore`), chỉ có trên máy đo.

In [ ]:
rows = []
for d in sorted(RAW.glob('SS-*')):
    steps = list(d.glob('ramp_*/run*/steps.json'))
    feats = sorted({p.parent.parent.name.replace('ramp_', '') for p in steps})
    rows.append({'thu_muc': d.name, 'so_ramp': len(steps), 'o (tinh nang_cuong do)': ', '.join(feats)[:110]})
display(pd.DataFrame(rows))

### B2.2 Một ramp mẫu: `login`, lần lặp 1

Mỗi ramp ghi `steps.json`: với từng bậc tải `target_rps` có p99 toàn hệ, p99 **riêng request tính năng**
(`feat_p99`), tỉ lệ lỗi, và cờ vi phạm. Tải tính năng = `target_rps × anchor%` (anchor của `login` là 10%,
chính là Δ trong mô hình — xem Phần C, hạn chế M5).

In [ ]:
def read_steps(path):
    j = json.load(open(path, encoding='utf-8'))
    return j, pd.DataFrame(j['steps'])

j, st = read_steps(RAW / 'SS-PROSP2/ramp_login_x1/run1/steps.json')
display(st[['target_rps', 'feat_rps', 'ok_per_s', 'p50', 'p99', 'feat_p99', 'err_rate', 'violated']].round(3))
print('diem gay:', j['breakpoint'])

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(st.target_rps, st.p99 * 1000, 'o-', label='p99 toan he')
ax.plot(st.target_rps, st.feat_p99 * 1000, 's--', label='p99 rieng request login')
ax.axhline(slo['p99_s'] * 1000, color='red', ls=':', label='SLO 250 ms')
ax.axvspan(j['breakpoint']['lo'], j['breakpoint']['hi'], color='orange', alpha=.2, label='khoang diem gay [lo, hi]')
ax.set_yscale('log'); ax.set_xlabel('tai (req/s)'); ax.set_ylabel('ms'); ax.legend(fontsize=8)
ax.set_title('Ramp login x1 / run1'); plt.tight_layout(); plt.show()

**Đọc biểu đồ:** độ trễ gần như phẳng rồi **tăng vọt** khi một node hết CPU — đó là lý do dùng ngưỡng mức sử
dụng CPU (`u*`) để dự đoán điểm gãy. Chú ý đường `login` đã vượt SLO *trước* toàn hệ: tuyến của chính tính năng
vỡ trước (mục B7).

### B2.3 Telemetry từng giây — `simple_metrics.csv`

Mỗi dòng là một giây. Với mỗi service `s` có `s_cpu` (đơn vị: % của 1 core, nên **trần C_s = 100 × cores**),
`s_workload` (request/giây tới service), các phân vị độ trễ. Cột `gt_*` là **nhãn thí nghiệm** (bậc tải,
tính năng, cờ khởi động `gt_step_warm`) — tách tiền tố để không bao giờ lẫn vào đặc trưng.

In [ ]:
sm = pd.read_csv(RAW / 'SS-PROSP2/ramp_login_x1/run1/simple_metrics.csv')
groups = {}
for c in sm.columns:
    key = 'nhan thi nghiem (gt_*)' if c.startswith('gt_') else ('he thong' if c in ('time', 'vm_cpu_util', 'vm_mem_avail_mb') else c.split('_')[0])
    groups.setdefault(key, []).append(c)
for k, v in groups.items():
    print(f'{k:24s} {v}')

SVC = FP.SCORED
cap = {s: 100 * cores[s] for s in SVC}
fig, ax = plt.subplots(figsize=(8, 3.4))
for s in SVC:
    ax.plot(sm['time'] - sm['time'].iloc[0], sm[f'{s}_cpu'] / cap[s], lw=1, label=s)
ax2 = ax.twinx(); ax2.step(sm['time'] - sm['time'].iloc[0], sm['gt_target_rps'], color='k', alpha=.4, where='post')
ax.set_xlabel('giay'); ax.set_ylabel('u = CPU / tran'); ax2.set_ylabel('tai (req/s)')
ax.legend(fontsize=7, ncol=5, loc='upper left'); ax.set_title('Muc su dung tung node theo thoi gian (login x1)')
plt.tight_layout(); plt.show()

Front-end (gateway) luôn có `u` cao nhất — dưới RE2, **front-end là node nghẽn ở gần như mọi ô**. Đây là sự
thật quan trọng nhất để hiểu vì sao đồ thị ít được "thử thách" (Phần C, G1).

## B3. Đồ thị nhân quả

Có **hai tầng đồ thị** — rất hay bị nhầm:

1. **Đồ thị phụ thuộc** (`src/graph/sockshop_agent_graph.json`): ai gọi ai. Là *cấu trúc*, trích từ kiến trúc.
2. **Mô hình nhân quả cấu trúc (SCM)** dùng để dự đoán: biến nào quyết định biến nào (tải → workload → CPU →
   mức sử dụng → phán quyết). Đồ thị phụ thuộc quyết định *cạnh nào được phép tồn tại* trong SCM.

### B3.1 Đồ thị phụ thuộc

In [ ]:
G = json.load(open(ROOT / 'src/graph/sockshop_agent_graph.json', encoding='utf-8'))
g = nx.DiGraph(); color = {'gateway': '#d62728', 'backend': '#1f77b4', 'database': '#7f7f7f',
                           'message_queue': '#9467bd', 'worker': '#8c564b'}
for n in G['nodes']: g.add_node(n['id'], t=n.get('type'))
for e in G['edges']: g.add_edge(e['source'], e['target'])
pos = {'front-end': (0, 0), 'session-db': (-1.3, -1), 'catalogue': (-2.6, 1.5), 'carts': (-.9, 1.5), 'user': (.9, 1.5),
       'orders': (2.6, 1.5), 'catalogue-db': (-2.6, 2.7), 'carts-db': (-.9, 2.7), 'user-db': (.9, 2.7), 'orders-db': (2.3, 2.7),
       'payment': (3.4, 2.7), 'shipping': (4.3, 1.5), 'rabbitmq': (4.3, 2.7), 'queue-master': (5.4, 2.7)}
fig, ax = plt.subplots(figsize=(9, 4.2))
nx.draw_networkx(g, pos, ax=ax, node_color=[color[g.nodes[n]['t']] for n in g], node_size=1500,
                 font_size=7, font_color='white', arrowsize=12, edge_color='#555')
ax.set_title('Do thi phu thuoc Sock Shop (do = gateway, xanh = backend, xam = DB)'); ax.axis('off'); plt.show()

Mọi request đi qua **front-end** (gateway duy nhất). `orders` là node "quạt ra" lớn nhất: một lần đặt hàng gọi
`user`, `carts`, `payment`, `shipping`.

### B3.2 SCM dùng cho dự đoán

Với tải nền `L` (req/s tại gateway) và một tính năng có tỉ lệ traffic Δ, chuỗi gọi `chain` và bội số gọi `k_s`:

$$
\underbrace{L,\ \Delta,\ \text{chain},\ k}_{\text{gốc — can thiệp } do(\cdot)}
\;\longrightarrow\; W_s \;\longrightarrow\; \text{CPU}_s = \alpha_s + \beta_s W_s
\;\longrightarrow\; u_s = \frac{\text{CPU}_s}{C_s}
\;\longrightarrow\; \text{phán quyết: } \max_s u_s \lessgtr u^*
$$

- **Workload nền:** `W_s = ρ_s · L` — ρ_s là tỉ lệ workload của service `s` so với gateway theo cơ cấu request nền.
- **Can thiệp:** tính năng mới là một phép `do(Δ, chain, k)` ở **node gốc** (traffic vào từ ngoài hệ). Vì can
  thiệp nằm ở gốc, *truncated factorisation* không cắt cạnh nào — đó là lý do toán tử `do()` không cho lợi thế
  ước lượng so với hồi quy thường (kết quả RQ4 của bài), còn **đồ thị** thì có (quyết định tải đổ vào node nào).

In [ ]:
d = nx.DiGraph()
d.add_edges_from([('L (tai nen)', 'W_fe'), ('Delta, chain, k\n(can thiep)', 'W_fe'), ('Delta, chain, k\n(can thiep)', 'W_s'),
                  ('L (tai nen)', 'W_s'), ('W_fe', 'CPU_fe'), ('W_s', 'CPU_s'), ('CPU_fe', 'u_fe'), ('CPU_s', 'u_s'),
                  ('u_fe', 'max u vs u*'), ('u_s', 'max u vs u*'), ('max u vs u*', 'PHAN QUYET\n+ diem gay R*')])
p = {'L (tai nen)': (0, 1), 'Delta, chain, k\n(can thiep)': (0, -1), 'W_fe': (1.6, 1), 'W_s': (1.6, -1),
     'CPU_fe': (3.1, 1), 'CPU_s': (3.1, -1), 'u_fe': (4.5, 1), 'u_s': (4.5, -1), 'max u vs u*': (6, 0),
     'PHAN QUYET\n+ diem gay R*': (7.6, 0)}
fig, ax = plt.subplots(figsize=(10, 3.2))
nx.draw_networkx(d, p, ax=ax, node_size=2600, node_color='#cfe2f3', font_size=7, arrowsize=12)
ax.set_title('SCM cua bo du doan (moi node s trong 5 node cham diem)'); ax.axis('off'); plt.show()

### B3.3 Cạnh *học* được (backpressure) — có nhưng **không** dùng trong bộ dự đoán khả thi

`src/graph/sockshop_scm_edges.json` là kết quả chọn cạnh CPU→CPU giữa service (theo CausIL: chỉ cạnh được tri thức
miền cho phép, phải qua chấm điểm held-out, knee-point và kiểm OOD). Các cạnh này thuộc đường `CapacityAgent`
cũ (RQ1–RQ4); bộ dự đoán khả thi ở mục B5 **chỉ** dùng cạnh workload theo chiều gọi.

In [ ]:
E = json.load(open(ROOT / 'src/graph/sockshop_scm_edges.json', encoding='utf-8'))
print('canh cuoi cung duoc giu:', E['final_edges'])
display(pd.DataFrame(E['scores']).sort_values('r2_gain', ascending=False))

### B3.4 Chuỗi gọi: taxonomy (viết tay) so với thực tế (đo bằng probe)

Mỗi tính năng được ánh xạ **trước khi probe** vào một *archetype*; archetype mang `chain` và Δ viết tay. Sau khi
agent mù cài xong, `probe_feature_chain.py` gọi tính năng ~20 lần **khi hệ rảnh** (không dùng dữ liệu tải) và đếm
số lần mỗi backend bị gọi → **bội số `k_s`** và **chain thật**.

In [ ]:
kdoc = {}
for f in ['k_measured_v3.json', 'k_measured_v4new.json']:          # v3 = 8 tinh nang vong 2, v4new = reorder/orderfull
    kdoc.update(json.load(open(FROZEN / f, encoding='utf-8'))['features'])
FEATS = ['login', 'register', 'wishlist', 'catsearch', 'account', 'preview', 'orderhist', 'related', 'reorder', 'orderfull']
rows = []
for f in FEATS:
    kf = kdoc[f]; arch = FP.FEATURE_ARCHETYPE[f]
    tax = SOCKSHOP_CALL_CHAINS[arch]['services']
    rows.append({'tinh_nang': f, 'archetype': arch, 'Delta_%': HARNESS.FEATURES[f]['anchor'],
                 'chain_taxonomy': tax, 'chain_do_duoc': kf['chain_measured'],
                 'khop?': set(tax) == set(kf['chain_measured']),
                 'k (>0)': {s: v for s, v in kf['measured_per_use'].items() if v > 0},
                 'Sum k backend': sum(v for s, v in kf['measured_per_use'].items() if s != 'front-end'),
                 'p99 luc ranh (ms)': round(1000 * kf['latency_idle']['p99'], 1) if 'latency_idle' in kf else None})
chains = pd.DataFrame(rows); display(chains)
print(f"chain taxonomy lech thuc te o {(~chains['khop?']).sum()}/{len(chains)} tinh nang")

Chain viết tay **lệch thực tế ở nhiều tính năng** (vd. `login` gọi thêm `carts` để merge giỏ). Đây là *dữ liệu*,
không phải lỗi: người cài mù không biết taxonomy. Bội số `k ≠ 1` (vd. `reorder`) là nguyên nhân sai số lớn nhất
đã tìm ra ở vòng độc lập — P3 sinh ra để sửa nó.

### B3.5 Quy trình probe đo `k` (chạy trên hệ thật, **không** dùng dữ liệu tải)

Probe chạy **sau khi agent mù cài xong, trước khi đóng băng P3**, khi hệ đang rảnh:

```
python experiments/collect/probe_feature_chain.py --features login,register,... \
    --n 20 --n-latency 200 --user-every 20 --save data/processed/frozen/k_measured_v2.json
```

1. Chụp bộ đếm request của từng backend (từ access-log).
2. Gọi tính năng `n = 20` lần, tuần tự.
3. `k_s` = (số request tăng thêm ở backend `s`) / 20; tách theo phương thức HTTP (`per_use_by_method`).
4. Gọi thêm 200 lần để lấy phân phối độ trễ **lúc rảnh** (`latency_idle`, dùng cho cổng nhu cầu phục vụ).

Notebook không chạy lại probe (cần hệ đang chạy và tính năng đã cài); nó đọc file probe đã lưu ở ô trên.

## B4. Cơ chế học từ dữ liệu (không có tính năng)

Với mỗi service `s`, từ `SS-TRAIN` (không trần CPU, bỏ bậc khởi động, tải ≤ 150 req/s):

- `ρ_s` = hồi quy qua gốc của `W_s` theo `W_gateway` → cơ cấu request nền.
- `CPU_s = α_s + β_s · W_s` bằng **NNLS** (α, β ≥ 0) → chi phí CPU mỗi request.

Ô dưới **fit lại từ đầu** rồi so với cơ chế trong file đã đóng băng — nếu khớp, chuỗi tái lập được.

In [ ]:
P2F = json.load(open(FROZEN / 'predictions_frozen_RE2_P2_prosp2.json', encoding='utf-8'))
mech_frozen = P2F['mechanism']
train = FP.select_train(FP.load_runs(str(RAW / 'SS-TRAIN')), P2F['params']['train_max_rps'])
mech_refit = FP.fit_mechanism(train)
m = pd.DataFrame(mech_frozen).T[['rho', 'alpha', 'beta', 'r2', 'n', 'w_max_train']]
m['beta_fit_lai'] = [mech_refit[s]['beta'] for s in m.index]
m['khop'] = np.isclose(m['beta'], m['beta_fit_lai'], rtol=1e-3)
display(m.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, s in zip(axes, ['front-end', 'carts', 'user']):
    ax.scatter(train[f'{s}_workload'], train[f'{s}_cpu'], s=3, alpha=.25)
    w = np.linspace(0, train[f'{s}_workload'].max(), 50); mm = mech_frozen[s]
    ax.plot(w, mm['alpha'] + mm['beta'] * w, 'r-', label=f"CPU = {mm['alpha']:.2f} + {mm['beta']:.3f}·W  (R²={mm['r2']:.3f})")
    ax.set_title(s); ax.set_xlabel('W (req/s)'); ax.set_ylabel('CPU (% 1 core)'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

Cơ chế tuyến tính khớp rất tốt **trong dải huấn luyện** (R² ≈ 0,99 ở front-end). Lưu ý: dữ liệu huấn luyện
**không có trần CPU**, còn dự đoán áp dụng dưới trần RE2 — đó là ngoại suy chế độ vận hành (Phần C, M4).

### B4.1 Ngưỡng u*

`u*` = mức sử dụng của node nghẽn ở **bậc đạt SLO cuối cùng** trong các ramp `base` (trung vị), chỉ đọc ramp không
có tính năng:

In [ ]:
# freeze_predictions.py con kiem CO CHE tren DU LIEU GIU LAI (tai > train_max) va luu ket qua vao file dong bang
display(pd.DataFrame(P2F['mechanism_check_holdout']).T if isinstance(P2F['mechanism_check_holdout'], dict)
        else pd.DataFrame(P2F['mechanism_check_holdout']))

### B4.1 Hiệu chỉnh ngưỡng u* — làm lại từ đầu

Quy trình (`calibrate_u_star`): với mỗi ramp **base** của `SS-LIMITS` (chỉ ramp không có tính năng — hàm từ chối nếu
gặp ramp tính năng), lấy bậc đạt SLO cuối cùng `lo`, tính mức sử dụng trung bình mỗi node ở bậc đó (bỏ giây khởi
động), lấy node cao nhất. `u*` = trung vị qua các ramp. Ô dưới tính lại và so với file đóng băng; sau đó áp **cùng
quy trình** lên các ramp `base` của `SS-PROSP2` để thấy u* của phiên đo tiến cứu.

In [ ]:
import re
def u_star_from(ramp_root, cores_, only_plain=True):
    vals = []
    for sp in sorted((RAW / ramp_root / 'ramp_base').glob('run*/steps.json')):
        if only_plain and not re.fullmatch(r'run\d+', sp.parent.name):
            continue
        jj = json.load(open(sp, encoding='utf-8')); lo = jj['breakpoint']['lo']
        d_ = pd.read_csv(sp.parent / 'simple_metrics.csv')
        d_ = d_.rename(columns={c: c[3:] for c in d_.columns if c.startswith('gt_')})
        d_ = d_[(d_['step_warm'] == 0) & (d_['target_rps'] == lo)]
        u_ = {s: d_[f'{s}_cpu'].mean() / (100 * cores_[s]) for s in FP.SCORED}
        b_ = max(u_, key=u_.get); vals.append({'ramp': sp.parent.name, 'lo': lo, 'nut nghen': b_, 'u': round(u_[b_], 4)})
    return pd.DataFrame(vals)

cal_limits = u_star_from('SS-LIMITS', P2F['params']['cores'])
cal_prosp = u_star_from('SS-PROSP2', P2F['params']['cores'])
display(cal_limits)
print(f"u* tinh lai (SS-LIMITS, 3 ramp goc) = {cal_limits.u.median():.4f}  |  trong file dong bang = {P2F['params']['u_star']}")
extra = u_star_from('SS-LIMITS', P2F['params']['cores'], only_plain=False)
extra = extra[~extra.ramp.str.fullmatch(r'run\d+')]
print('2 ramp base DO THEM sau (Pha B, KHONG dung de hieu chinh):'); display(extra)
print(f"cung quy trinh tren {len(cal_prosp)} ramp base cua SS-PROSP2: u* = {cal_prosp.u.median():.4f} "
      f"(khoang {cal_prosp.u.min():.3f}-{cal_prosp.u.max():.3f})")

u* đóng băng dùng đúng 3 ramp `base` gốc (`run1–3`, điểm gãy 220). Hai ramp `base` đo thêm trong Pha B vỡ ở 160
với u ≈ 0,68–0,71 — dấu hiệu sớm của hiện tượng trôi hệ thống. Nếu u* của phiên tiến cứu thấp hơn 0,878 thì ngưỡng
đóng băng đã **quá lạc quan** cho phiên đó — khớp với chẩn đoán
ở mục B7 (SLO vỡ khi front-end mới ở u ≈ 0,70–0,80). *Chỉ dùng để chẩn đoán:* đổi u* bây giờ là phát triển, không phải
tiến cứu.

## B5. Bốn mô hình dự đoán

Tất cả dùng chung cơ chế ở mục B4, chỉ khác cách đổ tải tính năng vào `W_s`:

| Mô hình | Workload của tính năng | Ý tưởng |
|---|---|---|
| **P0** | `W_s = ρ_s·L·(1+Δ)` | tăng đều mọi node theo cơ cấu nền (hành vi `CapacityAgent` cũ) |
| **P1** | `W_s = L·(ρ_s + Δ·k_s·[s∈chain])`, k = 1 | chỉ node trong chain nhận thêm tải |
| **P1_ctrl** | như P1 nhưng chain **ngẫu nhiên** cùng cỡ | đối chứng: chain có giá trị thật không? |
| **P2** | backend: `+Δ·k_s·c_s`; gateway: `+Δ·(1 + x·Σk)` | thêm chi phí mỗi lần gọi `c_s` và chi phí điều phối gateway `x`, **fit trên promo/recs** |
| **P3** | như P2 nhưng `k_s` **đo bằng probe** | sửa giả định mỗi hop gọi đúng 1 lần |

Vì mọi thứ tuyến tính theo `L`, điểm gãy có **dạng đóng**: với mỗi node,
`L_s = (u*·C_s − α_s) / (β_s · W_s(L=1))`, và `R* = min_s L_s`.

Tham số P2 đã đóng băng:

In [ ]:
p2 = P2F['p2_params']
print('c_s (chi phi moi lan goi so voi request nen):', {k: round(v, 3) for k, v in p2['c'].items()})
print('x (chi phi dieu phoi gateway / moi lan goi backend):', round(p2['x'], 4))
P = FP.FeasibilityPredictor(mech_frozen, P2F['params']['cores'], P2F['params']['u_star'], feature_cost=p2)

### B5.0 Quy trình fit tham số P2 (`c_s`, `x`) — làm lại từ đầu

`experiments/feasibility/fit_feature_costs.py`, **chỉ trên tập phát triển** (ramp `promo`, `recs` của `SS-LIMITS`;
không đọc tập khoá `track`/`review`). Cơ chế nền, trần, u* lấy nguyên từ bản đóng băng gốc.

1. Với mỗi bậc tải **trước điểm gãy** và mỗi node: đo `W_s`, `CPU_s` thật.
2. **`c_s`** (backend trong chain) = CPU tăng thêm do tính năng ÷ (β_s × workload tăng thêm **đo được**) → chi phí **mỗi
   lần gọi** so với request nền (tách khỏi bội số `k`).
3. **`x`** (gateway): mỗi request tính năng tốn `(1 + x·n_calls)` lần một request nền; hồi quy qua gốc.

In [ ]:
import evaluate_frozen as EF, fit_feature_costs as FFC
FZ0 = json.load(open(FROZEN / 'predictions_frozen_RE2.json', encoding='utf-8'))
preds0 = {(p['predictor'], p['feature'], round(float(p['scale']), 3)): p for p in FZ0['predictions']}
with quiet():
    runs_dev = EF.load_ramps(str(RAW / 'SS-LIMITS'), EF.DEV)
rows_dev = FFC.build_rows(runs_dev, FZ0['mechanism'], FZ0['params']['cores'], preds0)
fit = FP.fit_feature_cost(rows_dev, FZ0['mechanism'])
saved = json.load(open(FROZEN / 'p2_params_dev.json', encoding='utf-8'))['params']
print(f"{len(runs_dev)} ramp phat trien, {len(rows_dev)} dong (buoc tai x node)")
display(pd.DataFrame({'fit lai': {**fit['c'], 'x': fit['x']}, 'file dong bang': {**saved['c'], 'x': saved['x']}}).round(4))
print('boi so chi phi gateway theo tinh nang:', {k: round(v, 3) for k, v in fit['gateway_multiple_by_feature'].items()},
      '| n_calls:', rows_dev.groupby('feature').n_calls.first().to_dict())

**Điểm yếu thấy ngay (M2):** cả hai tính năng phát triển đều có `n_calls = 3`, nên `x` được xác định từ **một** giá trị
`n_calls` duy nhất — dạng `1 + x·n_calls` không hề được kiểm; bất kỳ dạng nào đi qua điểm `n_calls = 3` đều fit như nhau.

### B5.1 Một phán quyết giải trình đầy đủ: `login` ở 150 req/s (P3)

`explain_text()` phơi bày từng bước tính — không tính lại gì ngoài chính `workloads()`/`cpu()`/`verdict()`.

In [ ]:
k_login = kdoc['login']['measured_per_use']
print(P.explain_text(150, mode='P2', feature='login', k=k_login))

### B5.2 Điểm gãy dạng đóng — nhìn bằng hình

Đường `u_s(L)` của từng node là **đường thẳng**; điểm gãy là chỗ đường cao nhất chạm `u*`.

In [ ]:
def plot_u(feature, k=None, chain=None, ax=None, title=''):
    Ls = np.linspace(20, 260, 60)
    for s in FP.SCORED:
        ax.plot(Ls, [P.utilization(L, mode='P2', feature=feature, k=k, chain=chain)[s] for L in Ls], label=s)
    r, node = P.breakpoint(mode='P2', feature=feature, k=k, chain=chain)
    ax.axhline(P.u_star, color='red', ls=':'); ax.axvline(r, color='k', ls='--')
    ax.set_title(f'{title}  R* = {r:.1f} ({node})', fontsize=9); ax.set_xlabel('L (req/s)'); ax.set_ylabel('u')
    return r

fig, axes = plt.subplots(1, 3, figsize=(13, 3.3), sharey=True)
plot_u('base', ax=axes[0], title='base')
plot_u('login', k=k_login, ax=axes[1], title='login P3 (chain taxonomy)')
plot_u('login', k=k_login, chain=kdoc['login']['chain_measured'], ax=axes[2], title='login P3+chain do duoc')
axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

> ⚠️ **Điều cần biết về P3 đã đóng băng.** `freeze_p3_prospective.py` gọi `breakpoint(mode='P2', k=k)` **không
> truyền `chain`**, nên P3 dùng **chain của taxonomy**; node ngoài chain taxonomy (vd. `carts` của `login`) không
> nhận tải dù `k = 1`. Trong khi đó trường `chain` ghi trong file đóng băng lại là `chain_measured`. Con số đóng
> băng (181,8) là đúng với code đã băm — chỉ nhãn trong file dễ đọc nhầm. Biến thể dùng chain đo được là
> **P3+chain** (hồi cứu, cột riêng trong `evaluate_p3.py`). Ô dưới kiểm chứng:

In [ ]:
chk = []
for f in FEATS:
    fz = json.load(open(FROZEN / f'predictions_frozen_RE2_P3_prosp_{f}.json', encoding='utf-8'))
    frozen_r = next(p['breakpoint_rps'] for p in fz['predictions'] if p['scale'] == 1.0)
    k = kdoc[f]['measured_per_use']
    r_tax, _ = P.breakpoint(mode='P2', feature=f, k=k)
    r_meas, _ = P.breakpoint(mode='P2', feature=f, k=k, chain=kdoc[f]['chain_measured'])
    chk.append({'tinh_nang': f, 'P3 dong bang': frozen_r, 'tinh lai (chain taxonomy)': round(r_tax, 1),
                'P3+chain do duoc': round(r_meas, 1)})
chk = pd.DataFrame(chk); display(chk)
assert (chk['P3 dong bang'] == chk['tinh lai (chain taxonomy)']).all(), 'KHONG tai lap duoc P3 dong bang!'
print('OK: tai lap bit-for-bit P3 dong bang cho ca 10 tinh nang')

### B5.3 Đóng băng — và kiểm tính toàn vẹn

Quy tắc bất khả xâm phạm: **agent mù cài → probe → đóng băng + băm SHA-256 → (commit git) → mới được chạy ramp**. Các
script đóng băng có chốt chặn: từ chối nếu thư mục ramp đã có dữ liệu của tính năng; từ chối nếu thiếu ánh xạ
archetype; bắt buộc `--p2-params` cho bộ tính năng mới. Mỗi file đóng băng ghi kèm SHA-256 của chính code dự đoán.

Ô dưới băm lại **mọi file đóng băng** và so với file `.sha256` đi kèm, rồi tra thời điểm git đầu tiên thấy file.

In [ ]:
import hashlib, subprocess
def sha(p): return hashlib.sha256(open(p, 'rb').read()).hexdigest()
def first_commit(p):
    r = subprocess.run(['git', 'log', '--diff-filter=A', '--format=%ci %h', '--', str(p.relative_to(ROOT))],
                       capture_output=True, text=True, cwd=ROOT).stdout.strip().splitlines()
    return r[-1] if r else '(chua commit)'
rows = []
for f in sorted(FROZEN.glob('predictions_frozen_*.json')):
    s = f.with_name(f.name + '.sha256')
    rec = open(s).read().split()[0] if s.exists() else None
    created = json.load(open(f, encoding='utf-8')).get('created')
    rows.append({'file': f.name, 'tao luc': created, 'git lan dau': first_commit(f),
                 'SHA khop .sha256?': (sha(f) == rec) if rec else 'khong co .sha256'})
integ = pd.DataFrame(rows); display(integ)
print('so file KHONG khop:', int((integ['SHA khop .sha256?'] == False).sum()))

**Đọc bảng:** mọi file khớp hash — nội dung không bị sửa kể từ lúc băm. Nhưng hãy so cột `git lần đầu` với ngày đo:
các file của vòng tiến cứu 2 (`prosp2`, `prosp3`, `prosp4`, `P3_prosp_*` trừ `browse`) được git thấy **lần đầu lúc
26/09 23:44**, tức **sau** khi ramp đã chạy (25–26/09). Các commit đóng băng gốc tạo trước ramp đã bị gỡ bằng
`git reset --soft` và không còn trong repo. Hash chứng minh nội dung không đổi, nhưng **không** chứng minh thời điểm;
trường `created` là tự khai. Với vòng 2, bằng chứng thời điểm độc lập hiện chỉ còn nhật ký làm việc trong `docs/`.
Vòng sau: commit file đóng băng **và push** (hoặc gửi hash cho bên thứ ba) trước khi chạy ramp.

### B5.4 Agent sản phẩm: `NewFeatureFeasibilityAgent.assess()`

Đây là thứ `orchestrator.assess(...)` gọi thật. Nó thêm ba lớp quanh bộ dự đoán:

1. **Trần CPU sống:** đọc giới hạn thật từ `docker inspect` (dự phòng `limits.json`).
2. **Khoảng bất định bootstrap:** lấy mẫu lại **dữ liệu huấn luyện gốc**, fit lại cơ chế B lần, tính lại điểm gãy.
3. **Cổng throttling:** nếu node nghẽn có hạn ngạch < 0,5 core (vùng CFS throttling đã thấy ở C1/C2) → trả
   **`UNDECIDED`**, không đoán một con số.

(Cờ `latency_risk` = Σk ≥ 4 là cổng nhu cầu phục vụ đã **thất bại** khi kiểm tiến cứu — giữ để minh bạch.)

In [ ]:
from feasibility_agent import NewFeatureFeasibilityAgent
fa = NewFeatureFeasibilityAgent()
with quiet():
    v = fa.assess('login', L_peak=150, k=kdoc['login']['measured_per_use'], n_bootstrap=60)
display(pd.Series({k: val for k, val in v.__dict__.items() if k != 'utilization_by_node'}).to_frame('login @ 150 req/s'))
print('muc su dung theo node:', v.utilization_by_node)

## B6. Kết quả tiến cứu n = 10

**Đáp án** của mỗi ô = trung vị `lo` qua các lần lặp (cường độ ×1). Hai mốc nền **không dùng đồ thị nào**:

- `const_base`: đoán mọi tính năng vỡ ở đúng điểm gãy của baseline (bỏ qua tính năng).
- `const_oracle`: hằng số **tốt nhất có thể** = trung vị của chính các đáp án — mốc *gian lận*, dùng làm cận dưới
  cho mọi mô hình hằng số. Thắng nó mới đáng kể.

Dự đoán lấy từ file đóng băng: P1/P2 ở `P2_prosp2` / `prosp3` / **`prosp4`**, P3 ở từng file `P3_prosp_<f>`.

In [ ]:
def measured(ramp_dir, feats):
    cells = {}
    for sp in sorted((RAW / ramp_dir).glob('ramp_*/run*/steps.json')):
        jj = json.load(open(sp, encoding='utf-8')); f = jj.get('feature', 'base')
        if f != 'base' and (f not in feats or round(jj['feature_pct'] / HARNESS.FEATURES[f]['anchor'], 3) != 1.0):
            continue
        cells.setdefault(f, []).append(jj['breakpoint']['lo'])
        HI.setdefault(f, []).append(jj['breakpoint']['hi'] or 1.25 * jj['breakpoint']['lo'])
    return cells

HI = {}

def frozen_preds(feats):
    out = {}
    for tag in ('prosp2', 'prosp3', 'prosp4'):
        for pr in json.load(open(FROZEN / f'predictions_frozen_RE2_P2_{tag}.json', encoding='utf-8'))['predictions']:
            if pr['feature'] in feats and pr['scale'] == 1.0:
                out.setdefault(pr['feature'], {})[pr['predictor']] = pr['breakpoint_rps']
    for f in feats:
        fz = json.load(open(FROZEN / f'predictions_frozen_RE2_P3_prosp_{f}.json', encoding='utf-8'))
        out[f]['P3'] = next(p['breakpoint_rps'] for p in fz['predictions'] if p['scale'] == 1.0)
    return out

cells = measured('SS-PROSP2', FEATS); pr = frozen_preds(FEATS)
base_lo = float(np.median(cells['base'])); meas = {f: float(np.median(cells[f])) for f in FEATS}
oracle = float(np.median(list(meas.values())))
res = pd.DataFrame([{'o': f, 'dap_an': meas[f], 'lan_lap': cells[f], 'P1': pr[f].get('P1'), 'P2': pr[f].get('P2'),
                     'P3': pr[f]['P3'], 'P3+chain': chk.set_index('tinh_nang').loc[f, 'P3+chain do duoc'],
                     'const_base': base_lo, 'const_oracle': oracle} for f in FEATS]).set_index('o')
MODELS = ['P1', 'P2', 'P3', 'P3+chain', 'const_base', 'const_oracle']
err = pd.DataFrame({mdl: 100 * (res[mdl] - res['dap_an']) / res['dap_an'] for mdl in MODELS})
print(f'baseline do duoc = {base_lo:g} req/s ({len(cells["base"])} ramp) | const_oracle = {oracle:g}')
display(res)
print('\nsai so co dau (%) -- DUONG = du doan MUON hon thuc te = KHA THI GIA (nguy hiem):')
display(err.round(1))

In [ ]:
summary = pd.DataFrame({'|sai so| TB %': err.abs().mean(), 'trung vi %': err.abs().median(), 'xau nhat %': err.abs().max(),
                        'kha thi gia (du doan > hi)': [int(sum(res[mdl][f] > np.median(HI[f]) for f in FEATS)) for mdl in MODELS]})
print('KHA THI GIA = du doan vuot trung vi hi (bac dau vi pham): he DA vo ma mo hinh van bao dat')
display(summary.round(1))

from scipy import stats
for a_, b_ in [('P3', 'const_base'), ('P3', 'const_oracle'), ('P2', 'P3'), ('P3', 'P3+chain')]:
    w = stats.wilcoxon(err[a_].abs(), err[b_].abs())
    print(f'Wilcoxon |sai so| {a_:9s} vs {b_:13s}: p = {w.pvalue:.4f}')

### B6.1 Sàn nhiễu — vì sao chưa kết luận được "mô hình nào tốt hơn"

Cùng một ô đo 3–4 lần, `lo` thường lệch **một bậc lưới** (lưới hình học ×1,25 → một bậc ≈ 25%). Nếu sai số mô
hình nhỏ hơn biên độ lặp lại, dữ liệu **không đủ phân giải** để xếp hạng mô hình.

In [ ]:
noise = pd.Series({f: 100 * (max(v) - min(v)) / np.median(v) for f, v in cells.items()}, name='bien do lap %')
print(f"san nhieu (trung vi bien do lap) = {noise.median():.1f}%  |  |sai so| TB cua P3 = {err['P3'].abs().mean():.1f}%")

fig, ax = plt.subplots(figsize=(9, 3.6))
x = np.arange(len(FEATS))
for i, f in enumerate(FEATS):
    ax.plot([i] * len(cells[f]), cells[f], 'o', color='#999', ms=5, alpha=.7)
ax.plot(x, res['dap_an'], '_', ms=25, color='k', label='dap an (trung vi)')
ax.plot(x, res['P3'], 'D', color='#1f77b4', label='P3 dong bang')
ax.plot(x, res['P2'], 's', color='#ff7f0e', mfc='none', label='P2')
ax.axhline(oracle, color='green', ls=':', label='const_oracle')
ax.set_xticks(x, FEATS, rotation=30); ax.set_ylabel('diem gay lo (req/s)'); ax.legend(fontsize=7, ncol=4)
ax.set_title('Cham xam = tung lan lap (the hien san nhieu)'); plt.tight_layout(); plt.show()

## B7. Chẩn đoán: mô hình sai ở node nào?

So **mức sử dụng đo được** từng node (trung bình mỗi bậc, bỏ giây khởi động) với **dự đoán** của P3 ở cùng mức
tải. Nếu sai số tập trung ở front-end thì vấn đề là chi phí gateway (M1); nếu ở backend thì là `c_s`/`k`.

In [ ]:
def measured_u(feature, ramp_dir='SS-PROSP2'):
    out = []
    for p in sorted((RAW / ramp_dir / f'ramp_{feature}_x1' if feature != 'base' else RAW / ramp_dir / 'ramp_base').glob('run*/simple_metrics.csv')):
        dd = pd.read_csv(p); dd = dd[dd['gt_step_warm'] == 0]
        g_ = dd.groupby('gt_target_rps')[[f'{s}_cpu' for s in FP.SCORED]].mean()
        g_.columns = FP.SCORED; out.append(g_ / pd.Series(cap))
    return pd.concat(out).groupby(level=0).mean()

def compare(feature):
    mu = measured_u(feature); k = kdoc[feature]['measured_per_use'] if feature != 'base' else None
    pu = pd.DataFrame({L: P.utilization(L, mode='P2', feature=feature, k=k) for L in mu.index}).T[FP.SCORED]
    return mu, pu

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharey=True)
for ax, f in zip(axes, ['base', 'login', 'catsearch', 'reorder']):
    mu, pu = compare(f)
    for s, c in zip(['front-end', 'user', 'carts', 'orders'], ['C0', 'C1', 'C2', 'C3']):
        ax.plot(mu.index, mu[s], 'o-', color=c, label=f'{s} do'); ax.plot(pu.index, pu[s], '--', color=c)
    ax.axhline(P.u_star, color='red', ls=':'); ax.axvline(meas.get(f, base_lo), color='k', lw=.8)
    ax.set_title(f'{f}  (lien = do, dut = P3)', fontsize=9); ax.set_xlabel('L')
axes[0].set_ylabel('u'); axes[0].legend(fontsize=6); plt.tight_layout(); plt.show()

In [ ]:
# sai so muc su dung (diem %) o front-end va node backend lon nhat, cac bac TRUOC diem gay
rows = []
for f in ['base'] + FEATS:
    mu, pu = compare(f); lim = meas.get(f, base_lo); mu, pu = mu[mu.index <= lim], pu[pu.index <= lim]
    e = 100 * (pu - mu)
    rows.append({'o': f, 'front-end (du - do)': e['front-end'].mean(),
                 'backend lech nhat': e.drop(columns='front-end').abs().mean().idxmax(),
                 'lech backend': e.drop(columns='front-end').abs().mean().max(),
                 'u front-end do tai lo': mu['front-end'].iloc[-1]})
display(pd.DataFrame(rows).set_index('o').round(2))

**Cách đọc:** cột `front-end (du - do)` **âm** nghĩa là mô hình *đánh giá thấp* chi phí gateway → đoán điểm gãy
**muộn** (khả thi giả). Cột `u front-end đo tại lo` thấp hơn u* = 0,878 nghĩa là SLO vỡ **trước khi CPU bão hoà**.

**Kết quả thật (quan trọng — sửa một nhận định trước đó):**

- Ở `login`/`register`, mô hình dự đoán CPU front-end **gần đúng** (lệch chỉ −1,7 đến −3,3 điểm %). Vậy sai số
  +17% của hai ô này **không** chủ yếu do chi phí gateway (M1) như từng nghĩ.
- Thứ lệch là **ngưỡng**: SLO vỡ khi front-end mới ở u ≈ 0,70–0,80, trong khi u* = 0,878 (hiệu chỉnh ở 3 ramp
  `base` cũ, lúc base vỡ ở 220). Nguồn sai số chính là **M3 — một ngưỡng u* chung** (và việc u* hiệu chỉnh trên
  một phiên đo khác), không phải cơ chế CPU.
- `reorder` thì ngược lại: mô hình **thổi phồng** chi phí (+30 điểm ở front-end, +40 ở `user` do k_user = 6 × c_user)
  → đoán sớm (an toàn) — tức `c_s` fit trên promo/recs không chuyển được sang tính năng gọi lặp nhiều (M2).

---
# PHẦN C — Hạn chế đã biết và hướng cải tiến

## C1. Nhánh B (bộ dự đoán khả thi)

### Đồ thị nhân quả (nhánh B)

| # | Vấn đề | Kiểm ở đâu trong notebook |
|---|---|---|
| G1 | Dưới RE2 front-end nghẽn ở gần như mọi ô → đồ thị (chọn đúng node nghẽn) ít bị thử thách; P0/P1/P1_ctrl gần như trùng | mục B2.3, 5.2 |
| G2 | P3 đóng băng dùng chain **taxonomy**; chain đo được chỉ có ở biến thể hồi cứu P3+chain | mục B5.2 (ô kiểm chứng), cột P3+chain ở mục B6 |
| G3 | Thiếu biến **trạng thái dữ liệu** (số đơn/khách trong DB): chi phí request phụ thuộc kích thước dữ liệu | bẫy DB phình: `base` 240 → 125 req/s |
| G4 | `do()` không cho lợi thế vì can thiệp luôn ở node gốc — giá trị nằm ở đồ thị, không ở phép toán can thiệp | mục B3.2 |

### Mô hình dự đoán (nhánh B)

| # | Vấn đề | Kiểm ở đâu |
|---|---|---|
| M1 | Chi phí gateway `x·Σk` có dạng tuỳ tiện — nhưng mục B7 cho thấy CPU front-end dự đoán **đã gần đúng** (lệch ≤ 3 điểm ở 9/10 ô); **không** phải nguồn sai số chính | mục B7 (cột front-end) |
| M2 | `c_s`, `x` fit trên **2** tính năng có cùng `n_calls = 3` → gần như không xác định được `x` | mục B5 |
| M3 | **Nguồn sai số chính:** một ngưỡng `u*` = 0,878 cho mọi tính năng, hiệu chỉnh trên 3 ramp của phiên đo cũ; thực tế SLO vỡ ở u ≈ 0,70–0,80 (0,61 với `reorder`) | mục B4.1, mục B7 (cột u tại lo) |
| M4 | Cơ chế học **không trần** rồi áp dưới trần RE2 → ngoại suy chế độ | mục B4 |
| M5 | Δ viết tay **và** harness bơm đúng Δ đó → sai số ước lượng Δ bằng 0 theo cách dựng thí nghiệm | mục B2.2 |

### Thứ tự cải tiến đề xuất

1. **Hạ sàn nhiễu** dưới sai số mô hình (lưới mịn quanh điểm gãy, ≥ 4 lần lặp, đặt lại DB) — không làm thì không
   xác nhận được cải tiến nào.
2. **Thay ngưỡng `u*` chung bằng tiêu chí độ trễ** — mô hình hàng đợi nhiều lớp (MVA) dự đoán thời gian đáp
   ứng của *chính request tính năng* so với SLO (sửa M3, nguồn sai số chính theo mục B7). Tối thiểu: hiệu chỉnh
   lại u* trên các ramp `base` của cùng phiên đo.
3. **Đo nhu cầu phục vụ `D_s` từng node bằng probe tốc độ thấp** thay cho `c_s`, `x` fit trên 2 tính năng (sửa M2,
   thấy rõ ở `reorder`); đó cũng là đầu vào cần cho MVA ở bước 2.
4. **Dùng chain đo được** (P3+chain) làm mặc định (sửa G2).
5. **Cấu hình có node nghẽn khác front-end** (hạn ngạch ≥ 0,5 core) để thật sự thử thách đồ thị (sửa G1).

Mọi biến thể mới làm ở đây là **phát triển**; đóng băng bằng script rồi đo trên tính năng mới mới được gọi là tiến cứu.

## C2. Nhánh A (Global Causal DAG)

| # | Vấn đề | Kiểm ở đâu |
|---|---|---|
| A-1 | RE2-SS **thiếu tín hiệu tải**: workload gần như không đổi, R² `CPU ~ workload` rất thấp → MAPE thấp không chứng minh gì | A1.1 |
| A-2 | R² held-out **âm** ở nhiều mô hình hai biến dù MAPE < 10% | A5 |
| A-3 | Nhiều cơ chế CPU/độ trễ bị gắn **POOR** (vd. `carts_latency-50` MAPE hàng trăm %) → kết quả của chúng bị đánh dấu `unstable` | A8 |
| A-4 | **Không có cạnh CPU → độ trễ**: mô hình không diễn tả được bão hoà CPU làm tăng độ trễ | A4 (hình DAG) |
| A-5 | `do()` không có lợi thế vì can thiệp luôn ở node gốc — giá trị nằm ở đồ thị | A6 |
| A-6 | Trần hàng đợi `C = 1,5 × P99(W)` là **giả định**, không định danh được từ dữ liệu chưa bão hoà | A4 |
| A-7 | Siêu cộng dồn (Mệnh đề 3) không thấy trên RE2-SS vì hệ số hàng đợi fit ra 0; trên hệ một gateway, hai can thiệp không thể cùng ở node gốc nên giả định của mệnh đề không thoả | A7 |
| A-8 | `simulate_intervention()` báo `% đổi` so với **trung bình dữ liệu** thay vì mô hình ở δ = 0 → vài node báo giảm khi tải tăng | A6 |

## C3. Quy trình

| # | Vấn đề | Kiểm ở đâu |
|---|---|---|
| Q-1 | Bằng chứng **thời điểm** đóng băng của vòng 2 (commit git trước ramp) đã mất do `git reset --soft`; chỉ còn hash + nhật ký | B5.3 |
| Q-2 | `assess_predictive_performance.py` không đọc `prosp4` → P1/P2 thiếu ở `reorder`/`orderfull` | B6 (notebook đã tính đủ) |